<a href="https://colab.research.google.com/github/rdelhibabu/Rabotnov-Kernel-Based_LungCT/blob/main/Rabotnov_Kernel_Based_LungCT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# A Fractional-Order Rabotnov Kernel-Based Enhancement Model for Lung CT
# Classification Using Deep Neural Networks
# ==============================================================================

import os
import math
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as transforms
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import numpy as np

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==============================================================================
# 1. Rabotnov Fractional Enhancement Module
# ==============================================================================
class RabotnovEnhancer:
    """
    Implements the Rabotnov fractional exponential enhancement for image tensors.
    As per Algorithm 1: delta=0.75, l=0.5, truncated at T=6 for convergence stability.
    """
    def __init__(self, delta=0.75, l=0.5, T=6):
        self.delta = delta
        self.l = l
        self.T = T
        self.gamma_base = math.gamma(1 + self.delta)

        # Precompute Rabotnov coefficients
        self.H_coeffs = {}
        for n in range(2, self.T + 1):
            gamma_n = math.gamma(n * (1 + self.delta))
            self.H_coeffs[n] = ( (self.l**(n - 1)) * self.gamma_base ) / gamma_n

    def enhance(self, image_tensor):
        # Image tensor is assumed to be normalized [0, 1]
        enhanced = image_tensor.clone()
        for n in range(2, self.T + 1):
            enhanced += self.H_coeffs[n] * (image_tensor ** n)
        # Clip to ensure valid pixel range after enhancement
        return torch.clamp(enhanced, 0.0, 1.0)

# ==============================================================================
# 2. Modified ResNet-18 Architecture
# ==============================================================================
class ModifiedResNet18(nn.Module):
    """
    ResNet-18 backbone modified to accept 1-channel (grayscale) input and output
    predictions for 4 lung CT categories.
    """
    def __init__(self, num_classes=4):
        super(ModifiedResNet18, self).__init__()
        # Load pretrained ResNet-18
        resnet = models.resnet18(pretrained=True)

        # Modify the first convolutional layer for single-channel grayscale input
        old_conv = resnet.conv1
        new_conv = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        # Average the RGB weights to preserve pretrained features
        new_conv.weight.data = old_conv.weight.data.mean(dim=1, keepdim=True)
        resnet.conv1 = new_conv

        # Extract features up to the Global Average Pooling layer
        self.features = nn.Sequential(*list(resnet.children())[:-1])

        # Define the custom classifier head with Dropout
        self.classifier = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Dropout(p=0.5), # 50% Dropout as specified in the paper
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1) # Flatten
        x = self.classifier(x)
        return x

# ==============================================================================
# 3. Focal Loss Function
# ==============================================================================
class FocalLoss(nn.Module):
    """
    Dynamically handles class imbalances using Gamma = 2.0
    """
    def __init__(self, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt)**self.gamma * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        return focal_loss.sum()

# ==============================================================================
# 4. Data Loading (Dummy Generator for Colab execution)
# ==============================================================================
class DummyCTDataset(Dataset):
    """
    Generates synthetic grayscale images to test the pipeline without downloading
    the real CT dataset.
    REPLACE THIS WITH `torchvision.datasets.ImageFolder` for actual training.
    """
    def __init__(self, num_samples=300):
        self.num_samples = num_samples

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        # Generate random 1-channel 256x256 image normalized between [0, 1]
        img = torch.rand(1, 256, 256)
        label = torch.randint(0, 4, (1,)).item()
        return img, label

# To use your REAL dataset, uncomment the following block:
"""
from torchvision.datasets import ImageFolder
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((256, 256)),
    transforms.ToTensor() # Automatically scales to [0, 1]
])
train_dataset = ImageFolder(root='path_to_train_folder', transform=transform)
val_dataset = ImageFolder(root='path_to_val_folder', transform=transform)
test_dataset = ImageFolder(root='path_to_test_folder', transform=transform)
"""

# Using Dummy Data for demonstration
train_dataset = DummyCTDataset(num_samples=315) # Matches paper support numbers
val_dataset = DummyCTDataset(num_samples=100)
test_dataset = DummyCTDataset(num_samples=315)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# ==============================================================================
# 5. Initialization
# ==============================================================================
model = ModifiedResNet18(num_classes=4).to(device)
enhancer = RabotnovEnhancer(delta=0.75, l=0.5, T=6)
criterion = FocalLoss(gamma=2.0)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

# ==============================================================================
# 6. Training Loop
# ==============================================================================
num_epochs = 20
best_model_wts = copy.deepcopy(model.state_dict())
best_acc = 0.0

print("\n--- Starting Training Process ---")
for epoch in range(num_epochs):
    print(f'Epoch {epoch+1}/{num_epochs}')
    print('-' * 10)

    for phase in ['train', 'val']:
        if phase == 'train':
            model.train()
            dataloader = train_loader
        else:
            model.eval()
            dataloader = val_loader

        running_loss = 0.0
        running_corrects = 0

        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            # Apply Rabotnov Preprocessing (Algorithm 1)
            with torch.no_grad():
                inputs = enhancer.enhance(inputs)

            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        if phase == 'train':
            scheduler.step()

        epoch_loss = running_loss / len(dataloader.dataset)
        epoch_acc = running_corrects.double() / len(dataloader.dataset)

        print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

        if phase == 'val' and epoch_acc > best_acc:
            best_acc = epoch_acc
            best_model_wts = copy.deepcopy(model.state_dict())
    print()

print(f'Training complete. Best Validation Accuracy: {best_acc:4f}')
model.load_state_dict(best_model_wts)

# ==============================================================================
# 7. Evaluation & Metrics Output
# ==============================================================================
print("\n--- Running Final Evaluation on Test Set ---")
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        # Apply Rabotnov enhancement
        inputs = enhancer.enhance(inputs)

        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Calculate metrics
acc = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, average='macro', zero_division=0)
recall = recall_score(all_labels, all_preds, average='macro', zero_division=0)
f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
cm = confusion_matrix(all_labels, all_preds)

print(f"Overall Test Accuracy : {acc:.4f}")
print(f"Macro Precision       : {precision:.4f}")
print(f"Macro Recall          : {recall:.4f}")
print(f"Macro F1-Score        : {f1:.4f}\n")

print("Confusion Matrix:")
print(cm)

# Save the weights for GitHub distribution
torch.save(model.state_dict(), 'rabotnov_resnet18_weights.pth')
print("\nModel weights saved as 'rabotnov_resnet18_weights.pth'")

Using device: cpu


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 78.1MB/s]



--- Starting Training Process ---
Epoch 1/20
----------
Train Loss: 0.8425 Acc: 0.2635
Val Loss: 0.8317 Acc: 0.2500

Epoch 2/20
----------
